# Image Captioning Demo

本笔记本用于可视化测试集部分图片上的生成效果和评估测试集上的 BLEU 分数。

## 1. 加载必要的库

In [ ]:
import json
from pathlib import Path
from PIL import Image

import torch
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

from models import Encoder, RNNDecoder, RNNDecoderWithAttention
from datasets import CaptionDataset, collate_fn
from train import decode_caption, Config, evaluate, load_models

配置模型路径：

In [ ]:
# 配置
# TODO: 请根据实际情况修改设备和输出目录
cfg = Config()
device = cfg.device
model_path = "<path/to/best_model.pt>"
output_dir = Path(cfg.output_dir)

# 选择最新的输出目录
if model_path.endswith(".pt"):
    print(f"使用指定的模型: {model_path}")
elif output_dir.exists():
    output_subdirs = [d for d in output_dir.iterdir() if (d / "best_model.pt").exists()]
    if output_subdirs:
        output_subdirs.sort(key=lambda x: x.name, reverse=True)
        latest_output_dir = output_subdirs[0]
        model_path = latest_output_dir / "best_model.pt"
        print(f"使用最新的模型: {model_path}")
    else:
        print("未找到输出目录，请先训练模型")
else:
    print("输出目录不存在，请先训练模型")

## 2. 加载测试数据集和保存的模型

加载测试数据集：

In [ ]:
# 加载测试数据集
test_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

test_dataset = CaptionDataset(
    cfg.dataset_dir, split="test", transform=test_transform, cache=False
)
print(f"测试集大小: {len(test_dataset):,} 个描述")
print(f"测试集图片数量: {len(test_dataset.img_names):,} 张")

vocab = test_dataset.vocab
rev_vocab = test_dataset.rev_vocab
vocab_size = len(vocab)


加载保存的模型：

In [ ]:
# 加载模型
encoder, decoder = load_models(vocab_size, cfg)

# 加载保存的模型权重
checkpoint = torch.load(model_path, map_location=device)

# 加载权重
try:
    encoder.load_state_dict(checkpoint["encoder"])
    decoder.load_state_dict(checkpoint["decoder"])
except RuntimeError as e:
    raise RuntimeError("state_dict 不匹配，请检查 config 配置") from e

# 设置为评估模式
encoder.eval()
decoder.eval()
print("模型加载完成")

## 3. 可视化模型输出

运行以下脚本，来查看标注模型在图片上的生成效果。

首先，随机选择5张图片：

In [ ]:
# 选择测试集中的部分图片进行可视化
import random

# 随机选择5张图片
sample_indices = random.sample(range(len(test_dataset.img_names)), 5)
print(f"选择的图片索引: {', '.join(map(str, sample_indices))}")


可视化模型输出：

In [ ]:
# 可视化生成效果
with torch.no_grad():
    for i, img_idx in enumerate(sample_indices):
        # 获取图片路径
        img_name = test_dataset.img_names[img_idx]
        img_path = test_dataset.img_dir / img_name

        # 加载原始图片（用于显示）
        original_img = Image.open(img_path)

        # 获取模型输入的图片
        # 找到该图片对应的第一个描述的索引
        cap_idx = img_idx * test_dataset.cpi
        model_input, _, _ = test_dataset[cap_idx]
        model_input = model_input.unsqueeze(0).to(device)

        # 生成描述
        features = encoder(model_input)
        generated_caption = decoder.generate_caption(features, vocab)
        decoded_caption = decode_caption(generated_caption[0], rev_vocab)

        # 获取真实描述
        real_captions = []
        for j in range(test_dataset.cpi):
            _, cap, _ = test_dataset[cap_idx + j]
            real_captions.append(decode_caption(cap, rev_vocab))

        # 显示图片和描述
        plt.figure(figsize=(6, 6))
        plt.imshow(original_img)
        plt.axis('off')
        
        # 在图片上方添加描述
        caption_text = f"Image {i + 1}: {img_name}"
        caption_text += f"\nGenerated: {decoded_caption}\nReal captions:\n"
        for j, real_cap in enumerate(real_captions):
            caption_text += f"{j + 1}. {real_cap}\n"
        
        # 添加标题，放在图片上方
        print(caption_text)
        title = f"{decoded_caption}"
        plt.title(title)
        plt.tight_layout()
        plt.show()

## 4. 计算测试集上的 BLEU-4 分数

运行以下脚本，计算标注模型在测试集上的 BLEU-4 分数：

In [ ]:
from torch.utils.data import DataLoader

test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_fn,
    pin_memory=True,
)

scores = evaluate(encoder, decoder, test_loader, vocab, cfg)
bleu4 = scores["bleu-4"]
print(f"BLEU-4: {bleu4:.4f}")

## 5. 总结

本笔记本展示了模型在测试集上的生成效果，包括：
1. 随机选择了5张测试图片
2. 显示了原始图片
3. 生成了图片描述
4. 展示了真实描述作为对比
5. 计算了测试集的BLEU-4分数

通过这些可视化结果，我们可以直观地评估模型的生成效果。